In [2]:
! python --version

Python 3.11.13


In [3]:
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth

else:
    # Only for Colab
    !pip install -q --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install -q sentencepiece protobuf datasets huggingface_hub hf_transfer
    !pip install -q --no-deps unsloth

In [ ]:
# Fix for unsloth import errors on Colab
! pip install -q transformers -U

In [4]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.7.8: Fast Gemma3 patching. Transformers: 4.54.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [5]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 8,
    lora_alpha = 8,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model.language_model` require gradients


In [6]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [7]:
from datasets import Dataset, load_dataset
ds = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k")
ds['train'][10]

{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.",
 'input': 'I have found that I have an allergy to leotensin. They have taken me off of everything....I found in the information that one side effect if the red skin lensions with a purple center. I was looking to see if there was anything I could do to help the spots go away. I am a dental hygienist and I will be working the next two days. Is there anything that would cover them, so the patients would not be aware???',
 'output': 'Cellophane You for contacting Chat Doctor. Allergic reaction takes some time to settle. In the meanwhile you can cover it over the body by wearing appropriate long clothes which can cover most of the body to hide it. If itching occurs then you can take cetirizine once daily to prevent itching. Hope this answers your question. If you have additional questions or follow-up questions then please do not hesitate in writing to us. Wishing you good health

In [8]:
def convert_dataset(input_dataset):
    converted_records = []

    for record in input_dataset:
        user_content = record['input']
        assistant_content = record['output']

        converted_record = {
            'conversations': [
                {
                    'content': user_content,
                    'role': 'user'
                },
                {
                    'content': assistant_content,
                    'role': 'assistant'
                }
            ]
        }

        converted_records.append(converted_record)

    converted_dataset = Dataset.from_list(converted_records)

    return converted_dataset


converted_dataset = convert_dataset(ds['train'])

In [9]:
from unsloth.chat_templates import standardize_data_formats

dataset = standardize_data_formats(converted_dataset)
dataset[10]

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/112165 [00:00<?, ? examples/s]

{'conversations': [{'content': 'I have found that I have an allergy to leotensin. They have taken me off of everything....I found in the information that one side effect if the red skin lensions with a purple center. I was looking to see if there was anything I could do to help the spots go away. I am a dental hygienist and I will be working the next two days. Is there anything that would cover them, so the patients would not be aware???',
   'role': 'user'},
  {'content': 'Cellophane You for contacting Chat Doctor. Allergic reaction takes some time to settle. In the meanwhile you can cover it over the body by wearing appropriate long clothes which can cover most of the body to hide it. If itching occurs then you can take cetirizine once daily to prevent itching. Hope this answers your question. If you have additional questions or follow-up questions then please do not hesitate in writing to us. Wishing you good health.',
   'role': 'assistant'}]}

In [10]:
def apply_chat_template(examples):
    texts = tokenizer.apply_chat_template(examples["conversations"])
    return { "text" : texts }

dataset = dataset.map(apply_chat_template, batched = True)
dataset[10]["text"]

Map:   0%|          | 0/112165 [00:00<?, ? examples/s]

'<bos><start_of_turn>user\nI have found that I have an allergy to leotensin. They have taken me off of everything....I found in the information that one side effect if the red skin lensions with a purple center. I was looking to see if there was anything I could do to help the spots go away. I am a dental hygienist and I will be working the next two days. Is there anything that would cover them, so the patients would not be aware???<end_of_turn>\n<start_of_turn>model\nCellophane You for contacting Chat Doctor. Allergic reaction takes some time to settle. In the meanwhile you can cover it over the body by wearing appropriate long clothes which can cover most of the body to hide it. If itching occurs then you can take cetirizine once daily to prevent itching. Hope this answers your question. If you have additional questions or follow-up questions then please do not hesitate in writing to us. Wishing you good health.<end_of_turn>\n'

In [14]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/112165 [00:00<?, ? examples/s]

In [21]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
5.619 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

In [ ]:
prompt = "I ate some soaked chickpeas this morning, and now I am feeling weird. I'm finding it hard to swallow food, and the inside of my mouth is swelling up."

input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")
outputs = model.generate(input_ids, max_new_tokens=512)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [12]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from trl import SFTTrainer, SFTConfig

FastLanguageModel.for_training(model)  # VERY IMPORTANT before fine-tuning

# Fine-tune
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,
        max_steps = 30,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
)
trainer.train()


Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/112165 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 112,165 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,901,248 of 4,314,980,720 (0.35% trained)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,5.925400
2,5.516900
3,5.725000
4,5.793700
5,4.878200
6,4.631800
7,4.037800
8,3.974500
9,3.729800
10,3.719600


["user\nI ate some soaked chickpeas this morning, and now I am feeling weird. I'm finding it hard to swallow food, and the inside of my mouth is swelling up.\nmodel\nHello! I am very sorry you are suffering! Since you ate the chickpeas this morning, so you can be sure, they are not the cause for your discomfort, as it will occur in an hour. However, you may experience this problem. Swelling is quite unpleasant to feel, however, I did think that you do not have that problem. You may experience this problem after eating the chickpeas. In addition, my opinion is, you may have the problem with your food. Therefore, I recommend staying off of the food, for about 1 hour to 2 hours. You may also try eating very small amounts of other foods, which are easy to swallow. My opinion is that you would have it done, because the food is not the issue. Also, you may experience it if you are allergic to some things that you are eating! Hi!"]


In [13]:
# After training — regenerate chat template
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(tokenizer, chat_template = "gemma-3")

# Now you're safe to generate again
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "I ate some soaked chickpeas this morning, and now I am feeling weird. I'm finding it hard to swallow food, and the inside of my mouth is swelling up.",
    }]
}]
text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

inputs = tokenizer([text], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=512)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

["user\nI ate some soaked chickpeas this morning, and now I am feeling weird. I'm finding it hard to swallow food, and the inside of my mouth is swelling up.\nmodel\nHello. It might be worth consulting a doctor to see what could be the reason for the swelling that you are experiencing."]


In [14]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is a stomach ulcer?",}]
}]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer

outputs = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 512,
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

print(tokenizer.batch_decode(outputs))

Hello there! Thank you for your question. A stomach ulcer, also known as a gastric ulcer, is a sore that develops on the lining of your stomach or the upper part of your small intestine. Ulcers are caused by damage to the mucous membrane lining the walls of these organs. This means that the inner layer is destroyed leaving an open sore. Here are the main causes of stomach ulcers. One of the main causes of a stomach ulcer is a bacterial infection caused by the bacteria *Helicobacter pylori* which leads to infection. It can then lead to inflammation and damage to the stomach lining, increasing the risk of ulcer formation.  This can be passed through eating contaminated foods and drinks, or through poor hygiene. The consumption of medication that inhibits the stomach's production of acid. An ulcer may also come from the prolonged use of medication such as proton pump inhibitors that stops the secretion of stomach acid, which can lead to stomach ulcer when it has eaten through the lining o

In [ ]:
# Local Save
model.save_pretrained("DoctorGemma-3")
tokenizer.save_pretrained("DoctorGemma-3")

# Local GGUF Save
# model.save_pretrained_gguf(
#     "DoctorGemma-3-4B-8bit-GGUF",
#     quantization_type = "Q8_0",
# )

In [18]:
# Load Local Save

if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "DoctorGemma-3",
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "What is Insulin resistance?",}]
}]

text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64,
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

Hi there .
There are so many health conditions and diseases that doctors can deal with and you are the right person to deal with. Insulin resistance is when the cells of your body are not responding to insulin, resulting in insulin resistance. insulin is made in the pancreas and in the liver, fat and kidneys. Insulin causes


In [19]:
# Push to Hub

# model.push_to_hub_merged(
#     "atharvataras/DoctorGemma-3-4B-4bit", tokenizer,
#     token = "hf_"
# )

In [20]:
# Push GGUf to Hub

# model.push_to_hub_gguf(
#     "gemma-3-finetune",
#     quantization_type = "Q8_0",
#     repo_id = "HF_ACCOUNT/gemma-finetune-gguf",
#     token = "hf_...",
# )

In [22]:
! pip freeze

absl-py==1.4.0
accelerate==1.9.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.12.14
aiosignal==1.4.0
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
altair==5.5.0
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.9.0
argon2-cffi==25.1.0
argon2-cffi-bindings==21.2.0
array_record==0.7.2
arviz==0.22.0
astropy==7.1.0
astropy-iers-data==0.2025.7.21.0.41.39
astunparse==1.6.3
atpublic==5.1
attrs==25.3.0
audioread==3.0.1
autograd==1.8.0
babel==2.17.0
backcall==0.2.0
backports.tarfile==1.2.0
beautifulsoup4==4.13.4
betterproto==2.0.0b6
bigframes==2.11.0
bigquery-magics==0.10.1
bitsandbytes==0.46.1
bleach==6.2.0
blinker==1.9.0
blis==1.3.0
blobfile==3.0.0
blosc2==3.6.1
bokeh==3.7.3
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.1
Brotli==1.1.0
build==1.2.2.post1
CacheControl==0.14.3
cachetools==5.5.2
catalogue==2.0.10
certifi==2025.7.14
cffi==1.17.1
chardet==5.2.0
charset-normalizer==3.4.2
chex==0.1.90
clarabel==0.11.1
click==8.2.1
cloudpathlib==0.21.1
clou